# Gratis Fiyat Analizi — Veri Yükleme ve Temizlik

**Proje:** Gratis'te Fiyat Davranışı Analizi: Tüketici için Veri Odaklı Alışveriş Rehberi  
**Notebook:** 01 — Veri Yükleme ve Temizlik

Bu notebook'un amacı, Gratis web sitesinden toplanan ham fiyat verisini analiz ve modelleme için temiz, tutarlı ve tekrar kullanılabilir bir veri setine dönüştürmektir.

## Bu Notebook'ta Yapılan İşlemler

1. SQLite veritabanından ham veri okunur.
2. Veri tipleri kontrol edilir.
3. Tarih sütunları `datetime` formatına çevrilir.
4. `begeni` sütunu sayısal formata dönüştürülür.
5. Eksik değerler anlamlarına göre düzenlenir.
6. Aynı ürünün aynı gün içindeki birden fazla gözleminden en güncel kayıt tutulur.
7. En az 5 gün gözlemlenen ürünler filtrelenir.
8. Çok kelimeli marka isimleri düzeltilir.
9. Temiz veri seti sonraki notebook'larda kullanılmak üzere kaydedilir.

## Çıktı Dosyası

Bu notebook sonunda aşağıdaki temiz veri dosyası oluşturulur:

`../data/processed/gratis_clean.csv`

In [21]:
# Veri işleme
import pandas as pd
import numpy as np

# Veritabanı bağlantısı
import sqlite3

# Dosya yolları
from pathlib import Path

# Uyarıları gizleme
import warnings
warnings.filterwarnings("ignore")

print("Kütüphaneler yüklendi ✓")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

Kütüphaneler yüklendi ✓
pandas: 3.0.2
numpy: 2.4.4


In [22]:
# === VERİTABANI YOLU ===
# Notebook, notebooks/ klasöründe olduğu için proje kökü bir üst dizindir.

PROJECT_ROOT = Path("..")
DB_PATH = PROJECT_ROOT / "data" / "gratis.db"

print(f"Veritabanı yolu: {DB_PATH}")
print(f"Veritabanı var mı?: {DB_PATH.exists()}")

# === VERİYİ OKU ===
conn = sqlite3.connect(DB_PATH)
df_raw = pd.read_sql_query("SELECT * FROM fiyat_gecmisi", conn)
conn.close()

# Genel görünüm
print(f"\nToplam kayıt sayısı: {len(df_raw):,}")
print(f"Sütun sayısı: {df_raw.shape[1]}")

print("\nSütunlar:")
print(df_raw.columns.tolist())

print("\nVeri tipleri:")
print(df_raw.dtypes)

Veritabanı yolu: ..\data\gratis.db
Veritabanı var mı?: True

Toplam kayıt sayısı: 168,356
Sütun sayısı: 14

Sütunlar:
['id', 'urun_id', 'isim', 'marka', 'kategori', 'fiyat', 'eski_fiyat', 'indirim_yuzde', 'kampanya', 'yorum_sayisi', 'begeni', 'url', 'tarih', 'kayit_zamani']

Veri tipleri:
id                 int64
urun_id              str
isim                 str
marka                str
kategori             str
fiyat            float64
eski_fiyat       float64
indirim_yuzde    float64
kampanya             str
yorum_sayisi       int64
begeni               str
url                  str
tarih                str
kayit_zamani         str
dtype: object


In [23]:
# === HAM VERİYE İLK BAKIŞ ===

display(df_raw.head())

print("=== EKSİK DEĞER SAYILARI ===")
print(df_raw.isnull().sum())

print("\n=== BENZERSİZ DEĞER ÖZETİ ===")
print(f"Benzersiz ürün sayısı: {df_raw['urun_id'].nunique():,}")
print(f"Benzersiz kategori sayısı: {df_raw['kategori'].nunique()}")
print(f"Benzersiz marka sayısı: {df_raw['marka'].nunique():,}")
print(f"Benzersiz tarih sayısı: {df_raw['tarih'].nunique()}")

print("\n=== TARİH ARALIĞI ===")
print(f"En eski tarih: {df_raw['tarih'].min()}")
print(f"En yeni tarih: {df_raw['tarih'].max()}")

,id,urun_id,isim,marka,kategori,fiyat,eski_fiyat,indirim_yuzde,kampanya,yorum_sayisi,begeni,url,tarih,kayit_zamani
0,1,10209726,Love Generation Lipstick Balm Wet Dream 07 Dar...,Love,Makyaj,229.0,578.0,60.4,250 TL ve Üzeri Alışverişe,7,2B,https://www.gratis.com/ruj/love-generation-lip...,2026-04-04 18:55,2026-04-04 19:28:27.003870
1,2,10215840,Loreal Paris Telescopic Extensionist Maskara,Loreal,Makyaj,679.5,1699.0,60.0,250 TL ve Üzeri Alışverişe,20,8B,https://www.gratis.com/maskara/loreal-paris-te...,2026-04-04 18:55,2026-04-04 19:28:27.003870
2,3,10201916,Maybelline New York Super Lock Brow Glue Kaş S...,Maybelline,Makyaj,520.0,1300.0,60.0,250 TL ve Üzeri Alışverişe,282,58B,https://www.gratis.com/kas-maskarasi/maybellin...,2026-04-04 18:55,2026-04-04 19:28:27.003870
3,4,10209065,Flormar Puffy Liquid Blush Likit Allık 002 Pea...,Flormar,Makyaj,280.0,700.0,60.0,250 TL ve Üzeri Alışverişe,51,13B,https://www.gratis.com/allik/flormar-puffy-liq...,2026-04-04 18:55,2026-04-04 19:28:27.003870
4,5,10212949,Flormar Volume Up Hacim ve Lifting Etkili Yüks...,Flormar,Makyaj,500.0,1250.0,60.0,250 TL ve Üzeri Alışverişe,142,23B,https://www.gratis.com/maskara/flormar-volume-...,2026-04-04 18:55,2026-04-04 19:28:27.003870


=== EKSİK DEĞER SAYILARI ===
id                    0
urun_id               0
isim                  0
marka                 0
kategori              0
fiyat                 0
eski_fiyat        17494
indirim_yuzde     17494
kampanya         129321
yorum_sayisi          0
begeni              201
url                   0
tarih                 0
kayit_zamani          0
dtype: int64

=== BENZERSİZ DEĞER ÖZETİ ===
Benzersiz ürün sayısı: 11,679
Benzersiz kategori sayısı: 14
Benzersiz marka sayısı: 442
Benzersiz tarih sayısı: 686

=== TARİH ARALIĞI ===
En eski tarih: 2026-04-04 18:55
En yeni tarih: 2026-05-06 20:27


In [24]:
# === MÜKERRER KAYIT KONTROLÜ ===
# Aynı ürün aynı tarih değerinde birden fazla kez kayıt edilmiş mi?

duplicate_count = df_raw.duplicated(subset=["urun_id", "tarih"]).sum()

print(f"Mükerrer (urun_id + tarih) kayıt sayısı: {duplicate_count:,}")
print(f"Toplam kayıt sayısı: {len(df_raw):,}")
print(f"Mükerrer oranı: %{duplicate_count / len(df_raw) * 100:.2f}")

if duplicate_count > 0:
    print("\nÖrnek mükerrer kayıtlar:")
    display(
        df_raw[df_raw.duplicated(subset=["urun_id", "tarih"], keep=False)]
        .sort_values(["urun_id", "tarih"])
        .head(10)
    )
else:
    print("\nMükerrer kayıt bulunmadı ✓")

Mükerrer (urun_id + tarih) kayıt sayısı: 0
Toplam kayıt sayısı: 168,356
Mükerrer oranı: %0.00

Mükerrer kayıt bulunmadı ✓


In [25]:
# === TEMİZLİK VE VERİ TİPİ DÖNÜŞÜMLERİ ===

df = df_raw.copy()

# 1) Tarih sütunlarını datetime formatına çevir
df["tarih"] = pd.to_datetime(df["tarih"], format="%Y-%m-%d %H:%M", errors="coerce")
df["kayit_zamani"] = pd.to_datetime(df["kayit_zamani"], errors="coerce")

# Gün bazında snapshot tarihi oluştur
df["snapshot_gun"] = df["tarih"].dt.normalize()

# 2) Beğeni sütununu sayısal hale getir
# Örnek dönüşümler:
# "383" -> 383
# "2B"  -> 2000
# "13B" -> 13000

def begeni_parse(value):
    if pd.isna(value):
        return np.nan
    
    s = str(value).strip().upper()
    
    if s.endswith("B"):
        try:
            return float(s[:-1].replace(",", ".")) * 1000
        except:
            return np.nan
    
    if s.endswith("M"):
        try:
            return float(s[:-1].replace(",", ".")) * 1_000_000
        except:
            return np.nan
    
    try:
        return float(s.replace(",", "."))
    except:
        return np.nan

df["begeni_sayi"] = df["begeni"].apply(begeni_parse)

# 3) İndirim yoksa boş gelen alanları anlamlı şekilde doldur
# eski_fiyat boşsa ürün indirimde değildir; bu durumda eski_fiyat = fiyat kabul edilir
df["eski_fiyat"] = df["eski_fiyat"].fillna(df["fiyat"])

# indirim_yuzde boşsa indirim yoktur; bu durumda 0 kabul edilir
df["indirim_yuzde"] = df["indirim_yuzde"].fillna(0)

# 4) İndirimde mi değişkeni oluştur
df["indirimde_mi"] = df["indirim_yuzde"] > 0

# 5) Kampanya boşluklarını düzenle
df["kampanya"] = df["kampanya"].fillna("Kampanya Yok")

# 6) Analizde kullanılmayacak teknik ID sütununu çıkar
df = df.drop(columns=["id"])

print("Temizlik tamamlandı ✓")
print(f"Satır sayısı: {len(df):,}")
print(f"Sütun sayısı: {df.shape[1]}")

print("\nVeri tipleri:")
print(df.dtypes)

print("\nEksik değerler:")
print(df.isnull().sum())

Temizlik tamamlandı ✓
Satır sayısı: 168,356
Sütun sayısı: 16

Veri tipleri:
urun_id                     str
isim                        str
marka                       str
kategori                    str
fiyat                   float64
eski_fiyat              float64
indirim_yuzde           float64
kampanya                    str
yorum_sayisi              int64
begeni                      str
url                         str
tarih            datetime64[us]
kayit_zamani     datetime64[us]
snapshot_gun     datetime64[us]
begeni_sayi             float64
indirimde_mi               bool
dtype: object

Eksik değerler:
urun_id            0
isim               0
marka              0
kategori           0
fiyat              0
eski_fiyat         0
indirim_yuzde      0
kampanya           0
yorum_sayisi       0
begeni           201
url                0
tarih              0
kayit_zamani       0
snapshot_gun       0
begeni_sayi      201
indirimde_mi       0
dtype: int64


## Ürün-Gün Bazına İndirgeme

Ham veri içinde aynı ürün aynı gün içerisinde birden fazla kez gözlemlenmiş olabilir. Zaman bazlı analizlerde her ürünün her gün için yalnızca bir gözleme sahip olması gerekir.

Bu nedenle aynı `urun_id` ve `snapshot_gun` kombinasyonlarında, `kayit_zamani` en güncel olan kayıt tutulmuştur.

In [26]:
# === ÜRÜN × GÜN BAZINA İNDİRGEME ===

print("İndirgeme öncesi satır sayısı:", f"{len(df):,}")

df_daily = (
    df.sort_values("kayit_zamani", ascending=False)
      .drop_duplicates(subset=["urun_id", "snapshot_gun"], keep="first")
      .sort_values(["urun_id", "snapshot_gun"])
      .reset_index(drop=True)
)

print("İndirgeme sonrası satır sayısı:", f"{len(df_daily):,}")
print("Çıkarılan satır sayısı:", f"{len(df) - len(df_daily):,}")

print("\nBenzersiz ürün sayısı:", f"{df_daily['urun_id'].nunique():,}")
print("Benzersiz gün sayısı:", df_daily["snapshot_gun"].nunique())

# Ürün başına kaç gün gözlem var?
gozlem_sayisi = df_daily.groupby("urun_id").size()

print("\n=== ÜRÜN BAŞINA GÖZLEM SAYISI DAĞILIMI ===")
print(gozlem_sayisi.describe())

print("\nSadece 1 gün gözlemlenen ürün sayısı:", f"{(gozlem_sayisi == 1).sum():,}")
print("5+ gün gözlemlenen ürün sayısı:", f"{(gozlem_sayisi >= 5).sum():,}")
print("10+ gün gözlemlenen ürün sayısı:", f"{(gozlem_sayisi >= 10).sum():,}")

İndirgeme öncesi satır sayısı: 168,356
İndirgeme sonrası satır sayısı: 121,282
Çıkarılan satır sayısı: 47,074

Benzersiz ürün sayısı: 11,679
Benzersiz gün sayısı: 15

=== ÜRÜN BAŞINA GÖZLEM SAYISI DAĞILIMI ===
count    11679.000000
mean        10.384622
std          3.372494
min          1.000000
25%          9.000000
50%         11.000000
75%         13.000000
max         15.000000
dtype: float64

Sadece 1 gün gözlemlenen ürün sayısı: 351
5+ gün gözlemlenen ürün sayısı: 10,628
10+ gün gözlemlenen ürün sayısı: 8,396


## Minimum Gözlem Filtresi

Ürün bazlı fiyat davranışı, indirim sıklığı ve fiyat oynaklığı gibi özelliklerin güvenilir hesaplanabilmesi için her ürünün yeterli sayıda gözleme sahip olması gerekir.

Bu nedenle en az **5 farklı gün** gözlemlenen ürünler analiz veri setinde tutulmuştur. Daha az gözlemlenen ürünler modelleme açısından güvenilir olmadığı için dışarıda bırakılmıştır.

In [27]:
# === YETERLİ GÖZLEMİ OLAN ÜRÜNLERİ FİLTRELE ===

MIN_GOZLEM = 5

yeterli_urunler = gozlem_sayisi[gozlem_sayisi >= MIN_GOZLEM].index

print("Filtreleme öncesi ürün sayısı:", f"{df_daily['urun_id'].nunique():,}")
print("Filtreleme sonrası ürün sayısı:", f"{len(yeterli_urunler):,}")
print("Atılan ürün sayısı:", f"{df_daily['urun_id'].nunique() - len(yeterli_urunler):,}")

df_clean = (
    df_daily[df_daily["urun_id"].isin(yeterli_urunler)]
    .copy()
    .reset_index(drop=True)
)

print("\nFiltreleme öncesi satır sayısı:", f"{len(df_daily):,}")
print("Filtreleme sonrası satır sayısı:", f"{len(df_clean):,}")

print("\n=== FİNAL TEMİZ VERİ — FİLTRELEME SONRASI ===")
print("Satır sayısı:", f"{len(df_clean):,}")
print("Ürün sayısı:", f"{df_clean['urun_id'].nunique():,}")
print("Kategori sayısı:", df_clean["kategori"].nunique())
print("Marka sayısı:", f"{df_clean['marka'].nunique():,}")
print("Gün sayısı:", df_clean["snapshot_gun"].nunique())
print("Tarih aralığı:", df_clean["snapshot_gun"].min().date(), "→", df_clean["snapshot_gun"].max().date())

print("\nÜrün başına ortalama gözlem:", round(df_clean.groupby("urun_id").size().mean(), 1), "gün")

Filtreleme öncesi ürün sayısı: 11,679
Filtreleme sonrası ürün sayısı: 10,628
Atılan ürün sayısı: 1,051

Filtreleme öncesi satır sayısı: 121,282
Filtreleme sonrası satır sayısı: 118,812

=== FİNAL TEMİZ VERİ — FİLTRELEME SONRASI ===
Satır sayısı: 118,812
Ürün sayısı: 10,628
Kategori sayısı: 14
Marka sayısı: 422
Gün sayısı: 15
Tarih aralığı: 2026-04-04 → 2026-05-06

Ürün başına ortalama gözlem: 11.2 gün


## Marka İsimlerinin Düzeltilmesi

Scraper kodunda marka bilgisi ürün adının ilk kelimesinden türetildiği için çok kelimeli marka adlarında eksik veya hatalı değerler oluşmuştur.

Örneğin:

- `Bee` → `Bee Beauty`
- `Golden` → `Golden Rose`
- `The` → `The Purest Solutions`
- `Note` → `Note Cosmetics`

Bu nedenle, Gratis marka sayfası ve ürün adları kontrol edilerek çok kelimeli marka isimleri düzeltilmiştir.

In [28]:
# === MARKA İSİMLERİNİ DÜZELTME ===

marka_duzeltme = {
    "Bee": "Bee Beauty",
    "Love": "Love Generation",
    "Vivienne": "Vivienne Sabo",
    "Some": "Some By Mi",
    "Hada": "Hada Labo Tokyo",
    "Body": "Body Fantasies",
    "Organic": "Organic Shop",
    "Planet": "Planet Essence",
    "Physicians": "Physicians Formula",
    "Beauty": "Beauty Bomb",
    "Influence": "Influence Beauty",
    "Hyp": "Hyp Me",
    "Miss": "Miss Kay",
    "Life": "Life In",
    "VT": "VT Cosmetics",
    "Much": "Much More Than",
    "360": "360 Hair Professional",
    "Ayouth": "Ayouth Veda",
    "Dina": "Dina Cosmetics",
    "Milton": "Milton Lloyd Essentials",
    "farm": "farm Rx",
    "Jeanne": "Jeanne Arthes",
    "Golden": "Golden Rose",
    "Note": "Note Cosmetics",
    "The": "The Purest Solutions"
}

# Düzeltme öncesi kontrol
onceki_marka_sayisi = df_clean["marka"].nunique()

# Mapping'i uygula
df_clean["marka"] = df_clean["marka"].replace(marka_duzeltme)

# Düzeltme sonrası kontrol
sonraki_marka_sayisi = df_clean["marka"].nunique()

print("Marka düzeltmesi tamamlandı ✓")
print(f"Düzeltme öncesi marka sayısı: {onceki_marka_sayisi}")
print(f"Düzeltme sonrası marka sayısı: {sonraki_marka_sayisi}")

print("\n=== DÜZELTME SONRASI TOP 15 MARKA ===")

top15_marka = (
    df_clean.groupby("marka")["urun_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(15)
)

for marka, sayi in top15_marka.items():
    print(f"{marka:<35} {sayi:>5} ürün")

Marka düzeltmesi tamamlandı ✓
Düzeltme öncesi marka sayısı: 422
Düzeltme sonrası marka sayısı: 422

=== DÜZELTME SONRASI TOP 15 MARKA ===
Eklips                                422 ürün
Beaulis                               417 ürün
Flormar                               394 ürün
Pastel                                366 ürün
Bee Beauty                            352 ürün
Nascita                               349 ürün
Note Cosmetics                        311 ürün
Loreal                                263 ürün
Maybelline                            204 ürün
Benri                                 198 ürün
LYKD                                  196 ürün
Nivea                                 188 ürün
Golden Rose                           163 ürün
Astra                                 146 ürün
The Purest Solutions                  126 ürün


In [29]:
# === FİNAL VERİ KALİTESİ KONTROLÜ ===

print("=== FİNAL VERİ SETİ ÖZETİ ===")
print(f"Satır sayısı: {len(df_clean):,}")
print(f"Ürün sayısı: {df_clean['urun_id'].nunique():,}")
print(f"Kategori sayısı: {df_clean['kategori'].nunique()}")
print(f"Marka sayısı: {df_clean['marka'].nunique():,}")
print(f"Snapshot günü: {df_clean['snapshot_gun'].nunique()}")
print(f"Tarih aralığı: {df_clean['snapshot_gun'].min().date()} → {df_clean['snapshot_gun'].max().date()}")

print("\n=== EKSİK DEĞER KONTROLÜ ===")
print(df_clean.isnull().sum())

print("\n=== MÜKERRER ÜRÜN-GÜN KONTROLÜ ===")
duplicate_daily = df_clean.duplicated(subset=["urun_id", "snapshot_gun"]).sum()
print(f"Mükerrer ürün-gün kaydı: {duplicate_daily}")

print("\n=== İNDİRİM ÖZETİ ===")
print(f"İndirimli gözlem sayısı: {df_clean['indirimde_mi'].sum():,}")
print(f"İndirimli gözlem oranı: %{df_clean['indirimde_mi'].mean() * 100:.1f}")
print(f"Ortalama indirim yüzdesi: %{df_clean.loc[df_clean['indirim_yuzde'] > 0, 'indirim_yuzde'].mean():.1f}")

=== FİNAL VERİ SETİ ÖZETİ ===
Satır sayısı: 118,812
Ürün sayısı: 10,628
Kategori sayısı: 14
Marka sayısı: 422
Snapshot günü: 15
Tarih aralığı: 2026-04-04 → 2026-05-06

=== EKSİK DEĞER KONTROLÜ ===
urun_id            0
isim               0
marka              0
kategori           0
fiyat              0
eski_fiyat         0
indirim_yuzde      0
kampanya           0
yorum_sayisi       0
begeni           127
url                0
tarih              0
kayit_zamani       0
snapshot_gun       0
begeni_sayi      127
indirimde_mi       0
dtype: int64

=== MÜKERRER ÜRÜN-GÜN KONTROLÜ ===
Mükerrer ürün-gün kaydı: 0

=== İNDİRİM ÖZETİ ===
İndirimli gözlem sayısı: 110,801
İndirimli gözlem oranı: %93.3
Ortalama indirim yüzdesi: %50.4


## Temiz Veri Setinin Kaydedilmesi

Temizlenen, gün bazına indirgenen, minimum gözlem filtresinden geçirilen ve marka isimleri düzeltilen veri seti sonraki notebook'larda kullanılmak üzere CSV formatında kaydedilmiştir.

CSV formatı tercih edilmiştir çünkü hem Jupyter/Pandas ile kolay okunur hem de GitHub ve ders teslimi için erişilebilir bir formattır.

In [30]:
# === TEMİZ VERİYİ KAYDET ===

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DIR / "gratis_clean.csv"

df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Temiz veri başarıyla kaydedildi ✓")
print(f"Dosya yolu: {OUTPUT_PATH}")

# Dosya boyutu
file_size_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)
print(f"Dosya boyutu: {file_size_mb:.2f} MB")

# Kaydedilen dosyayı tekrar okuyarak doğrula
df_check = pd.read_csv(
    OUTPUT_PATH,
    parse_dates=["tarih", "kayit_zamani", "snapshot_gun"],
    encoding="utf-8-sig"
)

print("\n=== KAYIT DOĞRULAMA ===")
print(f"Okunan satır sayısı: {len(df_check):,}")
print(f"Orijinal satır sayısı: {len(df_clean):,}")
print(f"Sütunlar eşleşiyor mu?: {list(df_check.columns) == list(df_clean.columns)}")
print(f"Snapshot tarih tipi: {df_check['snapshot_gun'].dtype}")

Temiz veri başarıyla kaydedildi ✓
Dosya yolu: ..\data\processed\gratis_clean.csv
Dosya boyutu: 33.29 MB

=== KAYIT DOĞRULAMA ===
Okunan satır sayısı: 118,812
Orijinal satır sayısı: 118,812
Sütunlar eşleşiyor mu?: True
Snapshot tarih tipi: datetime64[us]


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path('/Users/damlasimsek/Desktop/GRATİSVERİ/processed')
HAM_KLASOR = Path('/Users/damlasimsek/Desktop/GRATİSVERİ')

ham_dosyalar = [
    HAM_KLASOR / 'gratis_veriler_20260410_2243.csv',
    HAM_KLASOR / 'gratis_veriler_20260410_2249.csv',
    HAM_KLASOR / 'gratis_veriler_20260424_1951.csv',
    HAM_KLASOR / 'gratis_veriler_20260426_1419.csv',
    HAM_KLASOR / 'gratis_veriler_20260430_2237.csv',
]

parcalar = []
for dosya in ham_dosyalar:
    tmp = pd.read_csv(dosya, sep=';', encoding='utf-8-sig')
    parcalar.append(tmp)

ham_df = pd.concat(parcalar, ignore_index=True)
print(f'Ham veri birleştirildi: {len(ham_df):,} satır')

ham_df['tarih'] = pd.to_datetime(ham_df['tarih'])
ham_df['kayit_zamani'] = ham_df['tarih']
ham_df['snapshot_gun'] = ham_df['tarih'].dt.date
ham_df['tarih'] = ham_df['tarih'].dt.normalize()

def begeni_donustur(val):
    if pd.isna(val):
        return 0.0
    val = str(val).strip()
    if 'B' in val:
        try:
            return float(val.replace('B', '')) * 1000
        except:
            return 0.0
    try:
        return float(val)
    except:
        return 0.0

ham_df['begeni_sayi'] = ham_df['begeni'].apply(begeni_donustur)
ham_df['indirimde_mi'] = ham_df['indirim_yuzde'].notna() & (ham_df['indirim_yuzde'] > 0)
ham_df = ham_df.drop_duplicates(subset=['urun_id', 'snapshot_gun'])

print(f'Temizleme sonrası: {len(ham_df):,} satır')
print(f'Günler: {sorted(ham_df["snapshot_gun"].unique())}')

mevcut_clean = pd.read_csv(DATA_DIR / 'gratis_clean.csv', encoding='utf-8-sig', parse_dates=['tarih', 'snapshot_gun'])
yeni_clean = pd.concat([mevcut_clean, ham_df], ignore_index=True)

yeni_clean = yeni_clean.drop_duplicates(subset=['urun_id', 'snapshot_gun'])
yeni_clean = yeni_clean.sort_values(['urun_id', 'tarih']).reset_index(drop=True)

print(f'\nYeni clean veri: {len(yeni_clean):,} satır')
print(f'Günler: {sorted(yeni_clean["tarih"].dt.date.unique())}')

yeni_clean.to_csv(DATA_DIR / 'gratis_clean.csv', index=False, encoding='utf-8-sig')
print('\ngratis_clean.csv güncellendi ✓')

Ham veri birleştirildi: 26,987 satır
Temizleme sonrası: 21,928 satır
Günler: [datetime.date(2026, 4, 10), datetime.date(2026, 4, 24), datetime.date(2026, 4, 26), datetime.date(2026, 4, 30)]

Yeni clean veri: 140,740 satır
Günler: [datetime.date(2026, 4, 4), datetime.date(2026, 4, 5), datetime.date(2026, 4, 6), datetime.date(2026, 4, 7), datetime.date(2026, 4, 8), datetime.date(2026, 4, 9), datetime.date(2026, 4, 10), datetime.date(2026, 4, 11), datetime.date(2026, 4, 14), datetime.date(2026, 4, 15), datetime.date(2026, 4, 17), datetime.date(2026, 4, 19), datetime.date(2026, 4, 24), datetime.date(2026, 4, 26), datetime.date(2026, 4, 27), datetime.date(2026, 4, 30), datetime.date(2026, 5, 3), datetime.date(2026, 5, 4), datetime.date(2026, 5, 6)]

gratis_clean.csv güncellendi ✓


## Notebook Özeti

Bu notebook kapsamında Gratis fiyat verisi analiz ve modelleme için hazır hale getirilmiştir.

### Başlangıç Verisi

- Ham veri kaynağı: `gratis.db` SQLite veritabanı
- Başlangıç kayıt sayısı: **168,356**
- Başlangıç sütun sayısı: **14**

### Yapılan Temel İşlemler

1. SQLite veritabanından ham fiyat verisi okundu.
2. Tarih alanları `datetime` formatına dönüştürüldü.
3. `begeni` sütunu `"2B"`, `"13B"` gibi metin formatlarından sayısal değere çevrildi.
4. `eski_fiyat` ve `indirim_yuzde` boşlukları, "indirim yok" anlamına gelecek şekilde düzenlendi.
5. `kampanya` boşlukları `"Kampanya Yok"` olarak etiketlendi.
6. Aynı ürünün aynı gün içindeki birden fazla gözleminden en güncel kayıt tutuldu.
7. En az 5 farklı gün gözlemlenen ürünler filtrelendi.
8. Çok kelimeli marka adları düzeltildi.
9. Temiz veri seti `../data/processed/gratis_clean.csv` olarak kaydedildi.

### Final Veri Seti

- Temiz gözlem sayısı: **118,812**
- Benzersiz ürün sayısı: **10,628**
- Kategori sayısı: **14**
- Marka sayısı: **422**
- Snapshot günü: **15**
- Tarih aralığı: **2026-04-04 → 2026-05-06**
- Mükerrer ürün-gün kaydı: **0**
- İndirimli gözlem oranı: **%93.3**

### Sonraki Adım

Bu temiz veri seti, `02_kesifsel_analiz.ipynb` dosyasında keşifsel veri analizi için kullanılacaktır.